In [1]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from filterpy.kalman import KalmanFilter
import matplotlib.pyplot as plt
import logging
import glob
import os
import warnings
warnings.filterwarnings('ignore')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class RegimeSwitchModel:
    def __init__(self, symbol, interval='1d', train_pct=0.8, strategy='long-only'):
        self.symbol = symbol
        self.interval = interval
        self.train_pct = train_pct
        if strategy not in ['long-only', 'long-short']:
            raise ValueError("Strategy must be either 'long-only' or 'long-short'")
        self.strategy = strategy

    def compute_features(self, data, features_config):
        df = data.copy()
        for feature, config in features_config.items():
            if feature == 'log_return':
                df['log_return'] = np.log(df['Close']).pct_change()
            elif feature == 'lnrange':
                df['lnrange'] = np.log(df['High'] / df['Low'])
            elif feature == 'rsi':
                period = config.get('period', 14)
                delta = df['Close'].diff()
                gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
                loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
                rs = gain / loss
                df['rsi'] = 100 - (100 / (1 + rs))
                
        return df.dropna()
    
    def load_macro_data(self, macro_features):
        all_macro_data = []

        for feature in macro_features:
            print(f"Loading {feature} data...")
            macro_df = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/macro/fredData/{feature}.csv")
            macro_df['date'] = pd.to_datetime(macro_df['date'])
            
            # Check frequency by calculating median time delta
            time_deltas = macro_df['date'].diff().dropna()
            median_delta = time_deltas.median().days
            
            if median_delta > 1:
                logging.info(f"Feature {feature} has {median_delta}-day frequency. Last date: {macro_df['date'].max()}")
                
                # Resample to daily frequency
                macro_df.set_index('date', inplace=True)
                macro_df = macro_df.resample('D').ffill().bfill()
                macro_df.reset_index(inplace=True)
            
            all_macro_data.append(macro_df.set_index('date'))
            
        # Combine all macro features
        combined_macro_df = pd.concat(all_macro_data, axis=1)
        
        return combined_macro_df
    
    def load_glassnode_data(self):
        """
        Load and merge all available Glassnode metrics for the specified symbol.
        Returns a DataFrame with all metrics merged on the date index.
        """
        ticker_map = {
            "bitcoin": "BTC",
            "ethereum": "ETH",
            "solana": "SOL"
        }
        
        if self.symbol not in ticker_map:
            logging.warning(f"No Glassnode data available for: {self.symbol}")
            return pd.DataFrame()
            
        ticker = ticker_map[self.symbol]
        base_path = "/Users/valter.rebelo/MissionControl/data/onchainData"
        
        # Find all CSV files for the ticker
        files = glob.glob(f"{base_path}/{ticker}_*.csv")
        
        if not files:
            logging.warning(f"No Glassnode data files found for {self.symbol}")
            return pd.DataFrame()
        
        # Initialize with the first file to get the date index
        first_file = pd.read_csv(files[0])
        result = pd.DataFrame(index=pd.to_datetime(first_file['date']))
        
        # Process each file
        for file in files:
            try:
                # Read file
                df = pd.read_csv(file)
                
                # Get column names
                date_col = df.columns[0]  # First column (date)
                metric_col = df.columns[1]  # Second column (metric)
                
                # Create temporary DataFrame with just this metric
                temp_df = pd.DataFrame({
                    'date': pd.to_datetime(df[date_col]),
                    metric_col: df[metric_col]
                }).set_index('date')
                
                # Merge with result
                result = result.join(temp_df)
                
                logging.info(f"Loaded {metric_col} data for {self.symbol}")
                
            except Exception as e:
                logging.error(f"Error loading {file}: {str(e)}")
                continue
        
        logging.info(f"Loaded {len(result.columns)} Glassnode metrics for {self.symbol}")
        return result

    def load_data(self):
        try:
            df_1 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/candleData/{self.symbol}_candles.csv")
            df_1['date'] = pd.to_datetime(df_1['date'])
            df_1.set_index('date', inplace=True)

            df_2 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/assetData/{self.symbol}.csv")
            df_2['date'] = pd.to_datetime(df_2['date'])
            df_2.set_index('date', inplace=True)

            btc_df = pd.read_csv("/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv")
            btc_df['date'] = pd.to_datetime(btc_df['date'])
            btc_df.set_index('date', inplace=True)

            data = pd.merge(df_1, df_2[['total_volume', 'market_cap']], on='date', how='inner')
            data.rename(columns={'total_volume': 'Volume', 'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close'}, inplace=True)
            data.index.name = 'Date'

            if self.symbol != "bitcoin":
                data['close_btc'] = (data['Close'] / btc_df['close']) * 100
                data.dropna(inplace=True)

            if self.symbol == "bitcoin":
                data = data[data.index >= '2017-01-01']

            # Compute log_close for Kalman Filter
            data['log_close'] = np.log(data['Close'])

            # Compute dynamic features
            features_config = {
                'log_return': {},
                'lnrange': {},
                'rsi': {'period': 14}
            }

            #macro_features = ['move', 'treasury5YInflationExpectation', 'creditSpreads', 'vix']
            #macro_df = self.load_macro_data(macro_features)

            # For now, no features or weekly data is being computed for on-chain. 1. Check what's useful; 2. Update pipeline!!! 

            glassnode_data = self.load_glassnode_data()
            if not glassnode_data.empty:
                data = pd.merge(data, glassnode_data, left_index=True, right_index=True, how='left')

            # Merge macro data using index
            #data = pd.merge(data, macro_df, left_index=True, right_index=True, how='left')

            data = self.compute_features(data, features_config)

            if len(data) < 100:
                raise ValueError("Insufficient data points (<100) for meaningful analysis.")
            logging.info(f"Loaded {len(data)} rows for {self.symbol}")
            return data
        except Exception as e:
            logging.error(f"Data loading failed for {self.symbol}: {str(e)}")
            raise

    def split_data(self, data, embargo_percent=0.01):
        try:
            total_rows = len(data)
            train_end_idx = int(total_rows * self.train_pct)
            embargo_end_idx = int(train_end_idx + (total_rows * embargo_percent))
            
            train = data.iloc[:train_end_idx]
            embargo = data.iloc[train_end_idx:embargo_end_idx]
            test = data.iloc[embargo_end_idx:]
            
            self.train_data = train

            if len(train) < 50 or len(test) < 50:
                raise ValueError("Train or test set too small (<50 rows).")
            logging.info(f"Split data: train={len(train)}, embargo={len(embargo)}, test={len(test)}")
            return train, embargo, test
        except Exception as e:
            logging.error(f"Data splitting failed: {str(e)}")
            raise

    def normalize_features(self, train, test, features):

        train_mean = train[features].mean()
        train_std = train[features].std()
        train_normalized = (train[features] - train_mean) / train_std
        test_normalized = (test[features] - train_mean) / train_std

        return (pd.DataFrame(train_normalized, index=train.index, columns=features),
                pd.DataFrame(test_normalized, index=test.index, columns=features))

    def train_hmm(self, train, features):
        try:
            f_train_normalized, _ = self.normalize_features(train, train, features)
            hmm = GaussianHMM(n_components=3, covariance_type='diag', n_iter=500, random_state=42)
            hmm.fit(f_train_normalized)
            if not hmm.monitor_.converged:
                logging.warning("HMM training did not converge.")
            logging.info("HMM trained successfully")
            return hmm
        except Exception as e:
            logging.error(f"HMM training failed: {str(e)}")
            raise

    def train_kf(self, test):
        try:
            kf = KalmanFilter(dim_x=2, dim_z=1)
            kf.F = np.array([[1, 1], [0, 1]])
            kf.H = np.array([[1, 0]])
            kf.Q = np.eye(2) * 0.01
            kf.R = np.array([[10]])
            kf.x = np.array([test['log_close'].iloc[0], 0])
            kf.P = np.eye(2) * 1000
            mu, _, _, _ = kf.batch_filter(test['log_close'].values)
            kf_slope_raw = np.diff(mu[:, 0])
            kf_slope = np.concatenate([[0], kf_slope_raw])
            logging.info("Kalman Filter trained successfully")
            return mu[:, 0], kf_slope
        except Exception as e:
            logging.error(f"Kalman Filter training failed: {str(e)}")
            raise

    def resample_weekly(self, data):
        weekly = data.resample('W-MON').agg({
            'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 
            'Volume': 'sum', 'log_close': 'last', 'log_return': 'sum', 
            'lnrange': 'mean', 'rsi': 'mean'#, 'creditSpreads': 'last', 'move': 'last', 'treasury5YInflationExpectation': 'last', 'vix': 'last'
        })
        return weekly.dropna()

    def train_models_multi_res(self, train, test, features):
        try:
            f_train_daily, f_test_daily = self.normalize_features(train, test, features)
            hmm_daily = GaussianHMM(n_components=3, covariance_type='full', n_iter=50000, random_state=42)
            hmm_daily.fit(f_train_daily)
            kf_est_daily, kf_slope_daily = self.train_kf(test)

            train_weekly = self.resample_weekly(train)
            test_weekly = self.resample_weekly(test)
            f_train_weekly, f_test_weekly = self.normalize_features(train_weekly, test_weekly, features)
            hmm_weekly = GaussianHMM(n_components=3, covariance_type='full', n_iter=50000, random_state=42)
            hmm_weekly.fit(f_train_weekly)
            kf_est_weekly, kf_slope_weekly = self.train_kf(test_weekly)

            logging.info("Multi-resolution models trained successfully")
            return (hmm_daily, f_test_daily, kf_slope_daily), (hmm_weekly, f_test_weekly, kf_slope_weekly)
        except Exception as e:
            logging.error(f"Multi-resolution training failed: {str(e)}")
            raise

    def voting_machine(self, hmm_daily, f_test_daily, kf_slope_daily, 
                      hmm_weekly, f_test_weekly, kf_slope_weekly, test_daily):
        
        # Daily predictions
        hidden_states_daily = hmm_daily.predict(f_test_daily)
        state_means_daily = hmm_daily.means_[:, 0]
        fav_state_daily = np.argmax(state_means_daily)
        
        if self.strategy == 'long-only':
            hmm_daily_state = ['Long' if s == fav_state_daily else 'Flat' for s in hidden_states_daily]
            kf_daily_state = ['Long' if s > 0 else 'Flat' for s in kf_slope_daily]
        else:  # long-short
            hmm_daily_state = ['Long' if s == fav_state_daily else 
                              'Short' if s == np.argmin(state_means_daily) else 
                              'Flat' for s in hidden_states_daily]
            kf_daily_state = ['Long' if s > 0 else 'Short' if s < 0 else 'Flat' for s in kf_slope_daily]

        logging.info(f"Daily predictions: {len(hmm_daily_state)} states")

        # Weekly predictions
        hidden_states_weekly = hmm_weekly.predict(f_test_weekly)
        state_means_weekly = hmm_weekly.means_[:, 0]
        fav_state_weekly = np.argmax(state_means_weekly)
        
        if self.strategy == 'long-only':
            hmm_weekly_state = ['Long' if s == fav_state_weekly else 'Flat' for s in hidden_states_weekly]
            kf_weekly_state = ['Long' if s > 0 else 'Flat' for s in kf_slope_weekly]
        else:  # long-short
            hmm_weekly_state = ['Long' if s == fav_state_weekly else 
                               'Short' if s == np.argmin(state_means_weekly) else 
                               'Flat' for s in hidden_states_weekly]
            kf_weekly_state = ['Long' if s > 0 else 'Short' if s < 0 else 'Flat' for s in kf_slope_weekly]

        weekly_df_raw = pd.DataFrame({
            'hmm_weekly': hmm_weekly_state, 
            'kf_weekly': kf_weekly_state
        }, index=f_test_weekly.index)
        weekly_df = weekly_df_raw.reindex(test_daily.index[1:], method='ffill')
        
        logging.info(f"Weekly predictions: {len(hmm_weekly_state)} states, reindexed to {len(weekly_df)} rows")

        if len(hmm_daily_state) != len(weekly_df):
            logging.warning(f"Length mismatch: daily={len(hmm_daily_state)}, weekly={len(weekly_df)}. Using minimum length.")

        ens_state = []
        min_len = min(len(hmm_daily_state), len(weekly_df))
        
        for i in range(min_len):
            daily_long = hmm_daily_state[i] == 'Long' and kf_daily_state[i] == 'Long'
            daily_short = (self.strategy == 'long-short' and 
                         hmm_daily_state[i] == 'Short' and kf_daily_state[i] == 'Short')
            
            weekly_long = weekly_df['hmm_weekly'].iloc[i] == 'Long' and weekly_df['kf_weekly'].iloc[i] == 'Long'
            weekly_short = (self.strategy == 'long-short' and 
                          weekly_df['hmm_weekly'].iloc[i] == 'Short' and weekly_df['kf_weekly'].iloc[i] == 'Short')
            
            if daily_long and weekly_long:
                ens_state.append('Long')
            elif daily_short and weekly_short:
                ens_state.append('Short')
            else:
                ens_state.append('Flat')
                
        return pd.Series(ens_state, index=test_daily.index[1:min_len+1])

    def ensemble_predict(self, hmm, test, kf_slope, features):
        try:
            models_daily, models_weekly = self.train_models_multi_res(self.train_data, test, features)
            states = self.voting_machine(*models_daily, *models_weekly, test)
            logging.info("Ensemble prediction completed")
            return states
        except Exception as e:
            logging.error(f"Ensemble prediction failed: {str(e)}")
            raise

    def simulate_trading(self, test, states):
        try:
            # Strategy returns
            shifted_states = states.shift(1).fillna('Flat')
            position_multiplier = (shifted_states == 'Long').astype(int) - (shifted_states == 'Short').astype(int)
            returns = test['Open'].pct_change() * position_multiplier
            cum_returns = (1 + returns).cumprod()
            
            # Buy and hold returns
            bnh_returns = test['Close'].pct_change().fillna(0.0)
            bnh_cum_returns = (1 + bnh_returns).cumprod()
            
            result = pd.DataFrame({
                'Close': test['Close'], 
                'ens_state': states,
                'returns': returns, 
                'cum_returns': cum_returns,
                'bnh_returns': bnh_returns,
                'bnh_cum_returns': bnh_cum_returns
            })
            logging.info("Trading simulation completed")
            return result
        except Exception as e:
            logging.error(f"Trading simulation failed: {str(e)}")
            raise

    def evaluate(self, results):
        try:
            # Strategy evaluation
            logging.info(f"ens_state sample: {results['ens_state'].head().tolist()}")
            cum_returns = results['cum_returns'].fillna(1.0)
            logging.info(f"Final cum_returns: {cum_returns.iloc[-1]}, length: {len(results)}")
            returns = results['returns'].fillna(0.0)
            
            # Calculate strategy metrics
            ann_ret = (cum_returns.iloc[-1] ** (365/len(results))) - 1
            sharpe = (returns.mean() / returns.std()) * np.sqrt(365)
            # Calculate Sortino ratio (using negative returns only for denominator)
            neg_returns = returns[returns < 0]
            sortino = (returns.mean() / neg_returns.std()) * np.sqrt(365) if len(neg_returns) > 0 else np.inf
            ann_vol = returns.std() * np.sqrt(365)
            drawdowns = cum_returns / cum_returns.cummax() - 1
            max_dd = drawdowns.min()
            
            # Count strategy switches
            ens_state = results['ens_state']
            switches = sum(ens_state.iloc[i] != ens_state.iloc[i-1] for i in range(1, len(ens_state)))
            
            # Buy and hold metrics
            bnh_returns = results['bnh_returns']
            bnh_cum_returns = results['bnh_cum_returns']
            bnh_ann_ret = (bnh_cum_returns.iloc[-1] ** (365/len(results))) - 1
            bnh_sharpe = (bnh_returns.mean() / bnh_returns.std()) * np.sqrt(365)
            # Calculate benchmark Sortino
            bnh_neg_returns = bnh_returns[bnh_returns < 0]
            bnh_sortino = (bnh_returns.mean() / bnh_neg_returns.std()) * np.sqrt(365) if len(bnh_neg_returns) > 0 else np.inf
            bnh_ann_vol = bnh_returns.std() * np.sqrt(365)
            bnh_drawdowns = bnh_cum_returns / bnh_cum_returns.cummax() - 1
            bnh_max_dd = bnh_drawdowns.min()
            
            # Create metrics table
            metrics_df = pd.DataFrame({
                'Metric': ['Annualized Return', 'Sharpe Ratio', 'Sortino Ratio', 'Annualized Volatility', 
                          'Maximum Drawdown', 'Final Cum Return', 'Switches'],
                'Strategy': [ann_ret, sharpe, sortino, ann_vol, max_dd, cum_returns.iloc[-1], switches],
                'Buy & Hold': [bnh_ann_ret, bnh_sharpe, bnh_sortino, bnh_ann_vol, bnh_max_dd, 
                              bnh_cum_returns.iloc[-1], 'N/A']
            })
            metrics_df.set_index('Metric', inplace=True)
            
            logging.info(f"Evaluation metrics:\n{metrics_df}")
            return metrics_df
        except Exception as e:
            logging.error(f"Evaluation failed: {str(e)}")
            raise



In [ ]:
model = RegimeSwitchModel('bitcoin', strategy='long-short', train_pct=0.75)
data = model.load_data()
train, embargo, test = model.split_data(data)
train

In [ ]:
features = ['rsi']
states = model.ensemble_predict(None, test, None, features)
print("State Distribution:")
print(states.value_counts())

In [ ]:
results = model.simulate_trading(test, states)
metrics = model.evaluate(results)
print(results)


In [ ]:
import plotly.graph_objects as go

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create figure with secondary y-axis
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    vertical_spacing=0.1,
                    subplot_titles=(f"{model.symbol} Strategy vs Buy & Hold Returns", 
                                  "Market States"))

# Add returns traces on first subplot
fig.add_trace(go.Scatter(x=results.index, y=results['cum_returns'], 
                        mode='lines', name='Strategy Returns'),
              row=1, col=1)
fig.add_trace(go.Scatter(x=results.index, y=results['bnh_cum_returns'], 
                        mode='lines', name='Buy & Hold Returns'),
              row=1, col=1)

# Add states trace on second subplot
fig.add_trace(go.Scatter(x=states.index, y=states.astype('category').cat.codes,
                        mode='lines', name='Market State',
                        hovertext=states),
              row=2, col=1)

# Update layout
fig.update_layout(height=800,
                 showlegend=True)
fig.update_yaxes(title_text="Cumulative Returns", row=1, col=1)
fig.update_yaxes(title_text="State", row=2, col=1)
fig.update_xaxes(title_text="Date", row=2, col=1)

fig.show()

# Feature Exploration

In [6]:
model = RegimeSwitchModel('ethereum')
data = model.load_data()
data

2025-02-28 00:10:21,250 - INFO - Loaded price_realized_usd data for ethereum
2025-02-28 00:10:21,259 - INFO - Loaded futures_funding_rate_perpetual data for ethereum
2025-02-28 00:10:21,266 - INFO - Loaded mvrv_z_score data for ethereum
2025-02-28 00:10:21,270 - INFO - Loaded active_1y_2y data for ethereum
2025-02-28 00:10:21,275 - INFO - Loaded spot_cvd_sum data for ethereum
2025-02-28 00:10:21,282 - INFO - Loaded price_drawdown_relative data for ethereum
2025-02-28 00:10:21,287 - INFO - Loaded active_3m_6m data for ethereum
2025-02-28 00:10:21,290 - INFO - Loaded transfers_volume_exchanges_net_pit data for ethereum
2025-02-28 00:10:21,296 - INFO - Loaded profit_relative data for ethereum
2025-02-28 00:10:21,296 - INFO - Loaded 9 Glassnode metrics for ethereum
2025-02-28 00:10:21,304 - INFO - Loaded 1105 rows for ethereum


,Open,High,Low,Close,Volume,market_cap,close_btc,log_close,price_realized_usd,futures_funding_rate_perpetual,mvrv_z_score,active_1y_2y,spot_cvd_sum,price_drawdown_relative,active_3m_6m,transfers_volume_exchanges_net_pit,profit_relative,log_return,lnrange,rsi
Date,,,,,,,,,,,,,,,,,,,,
2022-02-15,2882.11,2961.88,2845.45,2935.65,1.418948e+10,3.817911e+11,6.885540,7.984684,1778.537790,0.000018,1.289776,1.886742e+07,41903.209576,-0.344382,1.009731e+07,86680.820402,0.821237,0.002000,0.040103,59.816755
2022-02-16,2936.08,3190.65,2919.97,3179.30,1.312059e+10,3.760719e+11,7.132633,8.064416,1779.527980,0.000052,1.241009,1.888180e+07,-11455.445163,-0.356944,1.005332e+07,50174.404312,0.811102,0.009986,0.088651,63.730503
2022-02-17,3181.39,3182.15,3060.88,3128.64,1.624284e+10,3.452526e+11,7.100379,8.048354,1779.576162,-0.000018,1.006270,1.888878e+07,-71292.149162,-0.404515,1.001509e+07,-13611.691447,0.740657,-0.001992,0.038855,66.427062
2022-02-18,3126.25,3159.33,2868.08,2881.61,1.606405e+10,3.339976e+11,7.104036,7.966104,1778.849669,0.000052,0.919126,1.886792e+07,-6521.295403,-0.427241,1.002082e+07,-4276.277741,0.729581,-0.010219,0.096717,56.878526
2022-02-19,2889.15,2940.81,2765.67,2792.30,9.956230e+09,3.308147e+11,6.968033,7.934621,1777.744285,0.000002,0.899031,1.895192e+07,1808.722275,-0.430965,1.002861e+07,25869.038723,0.726404,-0.003952,0.061402,42.403033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-19,2743.35,2749.95,2612.31,2669.49,1.339859e+10,3.272482e+11,2.795395,7.889643,2102.989069,0.000022,0.481066,1.099837e+07,-28008.529605,-0.441082,7.669652e+06,-6963.956617,0.678134,-0.003381,0.051348,45.452679
2025-02-20,2671.69,2736.47,2656.53,2714.80,1.455003e+10,3.305175e+11,2.811662,7.906474,2103.180519,0.000028,0.499003,1.109524e+07,19055.594931,-0.436410,7.715464e+06,-28737.371944,0.685648,0.002133,0.029648,45.110443
2025-02-21,2716.35,2767.69,2712.37,2741.59,3.177038e+10,3.205083e+11,2.786622,7.916293,2101.414213,0.000138,0.437288,1.109410e+07,-14116.363025,-0.452478,7.889154e+06,-516977.380745,0.645133,0.001242,0.020190,53.937578


In [7]:
data.columns.tolist()

['Open',
 'High',
 'Low',
 'Close',
 'Volume',
 'market_cap',
 'close_btc',
 'log_close',
 'price_realized_usd',
 'futures_funding_rate_perpetual',
 'mvrv_z_score',
 'active_1y_2y',
 'spot_cvd_sum',
 'price_drawdown_relative',
 'active_3m_6m',
 'transfers_volume_exchanges_net_pit',
 'profit_relative',
 'log_return',
 'lnrange',
 'rsi']

In [4]:
import pandas as pd
import numpy as np
import ta
import plotly.express as px  # Added for interactive heatmap
from typing import Dict
import warnings
warnings.filterwarnings('ignore')

def explore_features(data: pd.DataFrame, verbose: bool = True) -> Dict[str, pd.DataFrame]:
    """
    Comprehensive feature exploration pipeline for technical, on-chain, and macro indicators.
    
    Args:
        data: Input DataFrame with OHLCV, on-chain metrics, and macro features
        verbose: Whether to print analysis progress and display visualizations
        
    Returns:
        Dictionary containing processed data and analysis results
    """
    # Create a copy to avoid modifying the original data
    df = data.copy()
    
    if verbose:
        print("\nStarting feature exploration pipeline...")
    
    # Ensure required columns exist
    required_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"DataFrame must contain all required columns: {required_columns}")

    # 1. Technical Indicators
    if verbose:
        print("Calculating Technical Indicators...")
    
    # Momentum Indicators
    df['rsi_14'] = ta.momentum.RSIIndicator(close=df['Close'], window=14).rsi()
    df['rsi_7'] = ta.momentum.RSIIndicator(close=df['Close'], window=7).rsi()
    stoch = ta.momentum.StochasticOscillator(high=df['High'], low=df['Low'], close=df['Close'])
    df['stoch_k'] = stoch.stoch()
    df['stoch_d'] = stoch.stoch_signal()
    macd = ta.trend.MACD(close=df['Close'])
    df['macd'] = macd.macd()
    df['macd_signal'] = macd.macd_signal()
    df['macd_diff'] = macd.macd_diff()

    # Trend Indicators
    df['sma_7'] = ta.trend.SMAIndicator(close=df['Close'], window=7).sma_indicator()
    df['sma_30'] = ta.trend.SMAIndicator(close=df['Close'], window=30).sma_indicator()
    df['sma_365'] = ta.trend.SMAIndicator(close=df['Close'], window=365).sma_indicator()
    df['adx'] = ta.trend.ADXIndicator(high=df['High'], low=df['Low'], close=df['Close']).adx()

    # Volatility Indicators
    bb = ta.volatility.BollingerBands(close=df['Close'])
    df['bb_upper'] = bb.bollinger_hband()
    df['bb_middle'] = bb.bollinger_mavg()
    df['bb_lower'] = bb.bollinger_lband()
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']
    df['atr'] = ta.volatility.AverageTrueRange(high=df['High'], low=df['Low'], 
                                              close=df['Close']).average_true_range()

    # Volume Indicators
    df['volume_ma_30'] = df['Volume'].rolling(window=30).mean()
    df['volume_ratio'] = df['Volume'] / df['volume_ma_30']
    df['obv'] = ta.volume.OnBalanceVolumeIndicator(close=df['Close'], 
                                                  volume=df['Volume']).on_balance_volume()

    # Custom Price Features
    if verbose:
        print("Calculating Custom Price Features...")
    
    df['log_return'] = np.log(df['Close']).diff()
    df['realized_vol_30'] = df['log_return'].rolling(window=30).std() * np.sqrt(365)
    df['high_low_range'] = (df['High'] - df['Low']) / df['Close']
    df['price_sma30_ratio'] = df['Close'] / df['sma_30']
    df['price_sma365_ratio'] = df['Close'] / df['sma_365']

    # 2. On-chain Metrics Processing
    if verbose:
        print("Processing On-chain Metrics...")
    
    price_normalized_metrics = ['price_realized_usd', 'mvrv_z_score', 'reserve_risk']
    return_metrics = ['hash_rate_mean', 'active_1y_2y', 'active_3m_6m', 'utxo_profit_count', 
                    'utxo_loss_count', 'net_realized_profit_loss', 'cdd_account_based', 
                    'transfers_volume_exchanges_net_pit', 'lth_realized_supply_density_5pct', 
                    'lth_realized_supply_density_10pct', 'lth_realized_supply_density_15pct', 
                    'sth_realized_supply_density_5pct', 'sth_realized_supply_density_10pct', 
                    'sth_realized_supply_density_15pct', 'unrealized_profit', 'unrealized_loss']
    ratio_metrics = ['net_unrealized_profit_loss_account_based', 'realized_profits_to_value_ratio', 
                    'puell_multiple', 'futures_funding_rate_perpetual', 'profit_relative', 
                    'price_drawdown_relative', 'dormancy_flow', 'sopr_less_155', 'ssr_oscillator']
        
    # Process on-chain metrics - look for lowercase columns
    onchain_columns = [col for col in df.columns if any(metric.lower() == col.lower() for metric in 
                      price_normalized_metrics + return_metrics + ratio_metrics)]
    
    for col in onchain_columns:
        # Base transformations for all metrics
        df[f'{col}_ma7'] = df[col].rolling(window=7).mean()
        df[f'{col}_ma30'] = df[col].rolling(window=30).mean()
        df[f'{col}_ma90'] = df[col].rolling(window=90).mean()
        df[f'{col}_ma180'] = df[col].rolling(window=180).mean()
        df[f'{col}_ma365'] = df[col].rolling(window=365).mean()
        df[f'{col}_ratio_7'] = df[col] / df[f'{col}_ma7']
        df[f'{col}_ratio_30'] = df[col] / df[f'{col}_ma30']
        df[f'{col}_ratio_90'] = df[col] / df[f'{col}_ma90']
        df[f'{col}_ratio_180'] = df[col] / df[f'{col}_ma180']
        df[f'{col}_ratio_365'] = df[col] / df[f'{col}_ma365']
        
    

        # Specific transformations based on metric type
        if col.lower() in [m.lower() for m in price_normalized_metrics]:
            df[f'{col}_price_ratio'] = df[col] / df['Close']
            
        if col.lower() in [m.lower() for m in return_metrics]:
            df[f'{col}_ret'] = df[col].pct_change()
            df[f'{col}_vol'] = df[f'{col}_ret'].rolling(window=30).std()
            
        if col.lower() in [m.lower() for m in ratio_metrics]:
            df[f'{col}_zscore'] = (df[col] - df[col].rolling(window=30).mean()) / df[col].rolling(window=30).std()

    # 3. Macro Features Processing
    if verbose:
        print("Processing Macro Features...")
    
    macro_features = []
    macro_columns = [col for col in df.columns if col.lower() in [f.lower() for f in macro_features]]
    
    for col in macro_columns:
        # Returns and volatility
        df[f'{col}_ret'] = df[col].pct_change()
        df[f'{col}_vol'] = df[f'{col}_ret'].rolling(window=30).std()
        
        # Moving averages and ratios
        df[f'{col}_ma7'] = df[col].rolling(window=7).mean()
        df[f'{col}_ma30'] = df[col].rolling(window=30).mean()
        df[f'{col}_ma90'] = df[col].rolling(window=90).mean()
        df[f'{col}_ma180'] = df[col].rolling(window=180).mean()
        df[f'{col}_ma365'] = df[col].rolling(window=365).mean()
        df[f'{col}_ratio_7'] = df[col] / df[f'{col}_ma7']
        df[f'{col}_ratio_30'] = df[col] / df[f'{col}_ma30']
        df[f'{col}_ratio_90'] = df[col] / df[f'{col}_ma90']
        df[f'{col}_ratio_180'] = df[col] / df[f'{col}_ma180']
        df[f'{col}_ratio_365'] = df[col] / df[f'{col}_ma365']

        # Z-scores
        df[f'{col}_zscore'] = (df[col] - df[col].rolling(window=30).mean()) / df[col].rolling(window=30).std()

    # Clean data
    df_clean = df.replace([np.inf, -np.inf], np.nan).dropna()

    # Feature Analysis
    if verbose:
        print("\nPerforming Feature Analysis...")
    
    # Update feature patterns to include all indicators
    feature_patterns = [
        '_ret', 'rsi_', 'stoch_', 'macd', 'sma_', 'ema_', 'bb_', 
        'volume_', 'realized_vol_', 'price_', '_ratio', 'adx', 'atr', 'obv',
        '_zscore', '_vol'
    ] + onchain_columns + macro_columns
    
    matching_columns = [col for col in df_clean.columns if any(pattern in col for pattern in feature_patterns)]
    features_to_analyze = list(dict.fromkeys(['log_return'] + matching_columns))

    # Calculate correlation matrix
    correlation_matrix = df_clean[features_to_analyze].corr()

    # Extract and sort correlations with log_return
    if 'log_return' in correlation_matrix.columns:
        correlations_with_returns = correlation_matrix['log_return']
        
        # Separate positive and negative correlations
        positive_corrs = correlations_with_returns[correlations_with_returns > 0].sort_values(ascending=False)
        negative_corrs = correlations_with_returns[correlations_with_returns < 0].sort_values(ascending=True)
        
        if verbose:
            print("\n**Positive Correlations with log_return (High to Low):**")
            print(positive_corrs.to_string())
            print("\n**Negative Correlations with log_return (Low to High):**")
            print(negative_corrs.to_string())
            
            # Summary of strongest indicators by category
            print("\n**Top 5 Technical Indicators:**")
            tech_corrs = correlations_with_returns[~correlations_with_returns.index.str.contains('|'.join(onchain_columns + macro_columns))]
            print(tech_corrs.abs().sort_values(ascending=False).head().to_string())
            
            print("\n**Top 5 On-chain Indicators:**")
            onchain_corrs = correlations_with_returns[correlations_with_returns.index.str.contains('|'.join(onchain_columns))]
            print(onchain_corrs.abs().sort_values(ascending=False).head().to_string())
            
            if macro_columns:
                print("\n**Top 5 Macro Indicators:**")
                macro_corrs = correlations_with_returns[correlations_with_returns.index.str.contains('|'.join(macro_columns))]
                print(macro_corrs.abs().sort_values(ascending=False).head().to_string())
    
    # Generate interactive heatmap
    if not correlation_matrix.empty and verbose:
        print("\nGenerating interactive correlation heatmap...")
        fig = px.imshow(correlation_matrix,
                       color_continuous_scale='RdBu',
                       zmin=-1,
                       zmax=1,
                       title='Feature Correlation Heatmap')
        fig.update_layout(width=1000, height=1000)
        fig.update_xaxes(tickangle=45)
        fig.show()

    results = {
        'processed_data': df_clean,
        'correlation_matrix': correlation_matrix,
        'positive_correlations': positive_corrs if 'log_return' in correlation_matrix.columns else pd.Series(dtype=float),
        'negative_correlations': negative_corrs if 'log_return' in correlation_matrix.columns else pd.Series(dtype=float)
    }

    if verbose:
        print("\nFeature exploration completed!")
    
    return results

In [ ]:
# Run the function
results = explore_features(data)

# Access results
processed_data = results['processed_data']

### First Filtering

In [8]:
def select_features(data: pd.DataFrame, correlation_threshold: float = 0.3, vif_threshold: float = 10) -> Dict[str, pd.DataFrame]:
    """
    Select features based on correlation threshold and VIF analysis.
    
    Args:
        data: DataFrame containing all features
        correlation_threshold: Minimum absolute correlation with log_return to keep feature
        vif_threshold: Maximum VIF value allowed before removing feature
        
    Returns:
        Dictionary containing selected features and analysis results
    """
    import statsmodels.api as sm
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    
    def calculate_vif(X):
        """Calculate VIF for each feature"""
        return pd.DataFrame({
            'feature': X.columns,
            'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        }).set_index('feature')
    
    # 1. Correlation-based filtering
    correlation_matrix = data.corr()
    correlations_with_returns = correlation_matrix['log_return'].abs()
    
    # Select features with correlation above threshold
    selected_features = correlations_with_returns[
        correlations_with_returns > correlation_threshold
    ].index.tolist()
    
    # Remove log_return from features to analyze
    if 'log_return' in selected_features:
        selected_features.remove('log_return')
    
    logging.info(f"Features selected after correlation threshold: {len(selected_features)}")
    
    # 2. Check multicollinearity using correlation matrix
    selected_correlation = correlation_matrix.loc[selected_features, selected_features]
    
    # Find highly correlated pairs
    corr_pairs = []
    for i in range(len(selected_features)):
        for j in range(i+1, len(selected_features)):
            if abs(selected_correlation.iloc[i,j]) > 0.7:  # Typical threshold for high correlation
                corr_pairs.append({
                    'feature1': selected_features[i],
                    'feature2': selected_features[j],
                    'correlation': selected_correlation.iloc[i,j]
                })
    
    # 3. VIF Analysis
    X = data[selected_features].copy()
    
    # Initial VIF calculation
    initial_vif = calculate_vif(X)
    logging.info("Initial VIF values:")
    logging.info(initial_vif)
    
    # Iteratively remove features with high VIF
    final_features = selected_features.copy()
    while True:
        vif_data = calculate_vif(data[final_features])
        max_vif = vif_data['VIF'].max()
        
        if max_vif < vif_threshold:
            break
            
        # Remove feature with highest VIF
        feature_to_remove = vif_data['VIF'].idxmax()
        final_features.remove(feature_to_remove)
        logging.info(f"Removed {feature_to_remove} with VIF {max_vif:.2f}")
        
        if len(final_features) < 2:
            logging.warning("Too few features remaining after VIF filtering")
            break
    
    # Final VIF calculation
    final_vif = calculate_vif(data[final_features])
    
    # 4. Create correlation heatmap for final features
    final_correlation = correlation_matrix.loc[final_features, final_features]
    
    # Generate heatmap
    fig = px.imshow(final_correlation,
                    color_continuous_scale='RdBu',
                    zmin=-1,
                    zmax=1,
                    title='Final Features Correlation Heatmap')
    fig.update_layout(width=800, height=800)
    fig.update_xaxes(tickangle=45)
    
    # Prepare results
    results = {
        'initial_features': selected_features,
        'final_features': final_features,
        'initial_vif': initial_vif,
        'final_vif': final_vif,
        'correlation_pairs': pd.DataFrame(corr_pairs),
        'final_correlation': final_correlation,
        'correlation_heatmap': fig
    }
    
    # Print summary
    logging.info(f"\nFeature Selection Summary:")
    logging.info(f"Initial features after correlation threshold: {len(selected_features)}")
    logging.info(f"Final features after VIF filtering: {len(final_features)}")
    logging.info("\nFinal selected features:")
    for f in final_features:
        logging.info(f"- {f}")
    
    return results

# Example usage:
def analyze_features(data: pd.DataFrame):
    """
    Complete feature analysis pipeline combining explore_features and select_features
    """
    # First run feature exploration
    exploration_results = explore_features(data)
    processed_data = exploration_results['processed_data']
    
    # Then run feature selection
    selection_results = select_features(
        processed_data, 
        correlation_threshold=0.3,
        vif_threshold=10
    )
    
    # Display results
    print("\nFeature Selection Results:")
    print("==========================")
    print(f"\nInitial number of features: {len(processed_data.columns)}")
    print(f"Final number of features: {len(selection_results['final_features'])}")
    
    print("\nFinal Features VIF:")
    print(selection_results['final_vif'])
    
    print("\nHighly Correlated Feature Pairs:")
    if not selection_results['correlation_pairs'].empty:
        print(selection_results['correlation_pairs'])
    else:
        print("No highly correlated pairs found in final features")
    
    # Show correlation heatmap
    selection_results['correlation_heatmap'].show()
    
    return selection_results

# Run the analysis
results = analyze_features(data)



Starting feature exploration pipeline...
Calculating Technical Indicators...
Calculating Custom Price Features...
Processing On-chain Metrics...
Processing Macro Features...

Performing Feature Analysis...

**Positive Correlations with log_return (High to Low):**
log_return                                     1.000000
rsi_7                                          0.465228
stoch_k                                        0.446831
profit_relative_ratio_7                        0.423175
rsi_14                                         0.349630
profit_relative_zscore                         0.346186
mvrv_z_score_ratio_7                           0.329729
price_drawdown_relative_zscore                 0.322666
price_sma30_ratio                              0.310857
profit_relative_ratio_30                       0.298127
mvrv_z_score_ratio_30                          0.265278
futures_funding_rate_perpetual_zscore          0.241628
profit_relative_ratio_90                       0.211372
profit_

2025-02-28 00:10:31,744 - INFO - Features selected after correlation threshold: 9
2025-02-28 00:10:31,762 - INFO - Initial VIF values:
2025-02-28 00:10:31,762 - INFO -                                          VIF
feature                                     
rsi_14                            637.932766
rsi_7                             281.245665
stoch_k                            31.460450
price_sma30_ratio                1103.240914
mvrv_z_score_ratio_7               17.583853
price_drawdown_relative_ratio_7   285.265278
price_drawdown_relative_zscore     16.315606
profit_relative_ratio_7           613.717892
profit_relative_zscore             13.301446
2025-02-28 00:10:31,766 - INFO - Removed price_sma30_ratio with VIF 1103.24
2025-02-28 00:10:31,769 - INFO - Removed rsi_14 with VIF 324.17
2025-02-28 00:10:31,772 - INFO - Removed profit_relative_ratio_7 with VIF 299.52
2025-02-28 00:10:31,775 - INFO - Removed rsi_7 with VIF 73.92
2025-02-28 00:10:31,777 - INFO - Removed price_drawdow


Feature exploration completed!


2025-02-28 00:10:31,794 - INFO - 
Feature Selection Summary:
2025-02-28 00:10:31,795 - INFO - Initial features after correlation threshold: 9
2025-02-28 00:10:31,795 - INFO - Final features after VIF filtering: 4
2025-02-28 00:10:31,795 - INFO - 
Final selected features:
2025-02-28 00:10:31,796 - INFO - - stoch_k
2025-02-28 00:10:31,796 - INFO - - mvrv_z_score_ratio_7
2025-02-28 00:10:31,796 - INFO - - price_drawdown_relative_zscore
2025-02-28 00:10:31,796 - INFO - - profit_relative_zscore



Feature Selection Results:

Initial number of features: 134
Final number of features: 4

Final Features VIF:
                                     VIF
feature                                 
stoch_k                         6.171660
mvrv_z_score_ratio_7            5.408477
price_drawdown_relative_zscore  8.374035
profit_relative_zscore          7.724273

Highly Correlated Feature Pairs:
                          feature1                        feature2  \
0                           rsi_14                           rsi_7   
1                           rsi_14                         stoch_k   
2                           rsi_14               price_sma30_ratio   
3                           rsi_14  price_drawdown_relative_zscore   
4                           rsi_14          profit_relative_zscore   
5                            rsi_7                         stoch_k   
6                            rsi_7               price_sma30_ratio   
7                            rsi_7  price_drawdown